# Synthetic Dataset Generator
Should run in Colab. See https://colab.research.google.com/drive/1wVdUQgNU04Ck3Si6PDj-7fS1UHxCKN7_#scrollTo=rHQmdCtKug_R

In [2]:
!pip install -q requests bitsandbytes transformers==4.57.6

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 121.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.7 MB/s eta 0:00:00


In [3]:
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig, TextStreamer
import torch
import gc

In [43]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

MINITRON = "nvidia/Mistral-NeMo-Minitron-8B-Instruct" # Takes a long time to load
SMOL = "HuggingFaceTB/SmolLM2-1.7B-Instruct" # This is a model specifically for generating sample data
SMOL3 = "HuggingFaceTB/SmolLM3-3B"

FILE_FORMATS = [".csv", ".md"]

In [5]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [48]:
system_prompt = """
You are an expert in generating synthetic datasets tailored to a given business case and user requirements.
If the user specifies output fields, stick to those, don't add more. Fields should not contain nested fields.
If a field is a list, don't add more than 5 items to the list.
Create a unique records. Use diverse values for the equal fields between records. Don't repeat yourself.
Only output valid JSONL without any comments.
"""

def get_user_prompt(business_case, fields, nr_records):
    prompt = f"The business case is: {business_case}. The fields are: {fields} \nGenerate {nr_records} rows of data in JSONL format.\n"
    return prompt

In [54]:
def ask_hf(model, user_prompt, quant=True, max_new_tokens=400):

  messages = [
        {"role": "system", "content": system_prompt + " /no_think"},
        {"role": "user", "content": user_prompt}
      ]

  tokenizer = AutoTokenizer.from_pretrained(model)
  tokenizer.pad_token = tokenizer.eos_token
  input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")

  attention_mask = torch.ones_like(input_ids, dtype=torch.long, device="cuda")
  streamer = TextStreamer(tokenizer)

  if quant:
    hf_model = AutoModelForCausalLM.from_pretrained(model, device_map="auto", quantization_config=quant_config).to("cuda")
  else:
    hf_model = AutoModelForCausalLM.from_pretrained(model).to("cuda")

  outputs = hf_model.generate(input_ids=input_ids, attention_mask=attention_mask, max_new_tokens=max_new_tokens, streamer=streamer)

  # _, _, after = tokenizer.decode(outputs[0]).partition("assistant<|end_header_id|>")
  # content = after.strip()

  del tokenizer, input_ids, hf_model, outputs
  gc.collect()
  torch.cuda.empty_cache()

  # return outputs;



In [53]:
business_case = "The business case is restaurants"
fields = "name_of_restaurant, address, type_of_restaurant, average_price_of_a_meal, year_of_opening"
user_prompt = get_user_prompt(business_case, fields, 6)

ask_hf(SMOL3, user_prompt, 1500)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 13 March 2026
Reasoning Mode: /think

## Custom Instructions


You are an expert in generating synthetic datasets tailored to a given business case and user requirements.
If the user specifies output fields, stick to those, don't add more. Fields should not contain nested fields.
If a field is a list, don't add more than 5 items to the list.
Create a unique records. Use diverse values for the equal fields between records. Don't repeat yourself.
Only output valid JSONL without any comments.

<|im_start|>user
The business case is: The business case is restaurants. The fields are: name_of_restaurant, address, type_of_restaurant, average_price_of_a_meal, year_of_opening 
Generate 6 rows of data in JSONL format.
<|im_end|>
<|im_start|>assistant
<think>
Okay, let's tackle this. The user wants a dataset for restaurants with specific fields: name_of_restaurant, address, type_of_restaurant, average_price_of_a_meal, yea

Tot hier getest

In [ ]:
def query_llm(model_name: str, user_prompt):
    try:
        model = MODELS[model_name]

        if "gpt" in model.lower():
            # response = ask_gpt(model, user_prompt)
            print('gpt not supported yet')

        elif model in HF_MODELS:
            response = ask_hf(model, user_prompt)

        else:
            raise ValueError(f"Unsupported model. Use one of {', '.join(MODELS.keys())}")

        lines = [line.strip() for line in response.strip().splitlines() if line.strip().startswith("{")]

        return [json.loads(line) for line in lines]

    except Exception as e:
        raise Exception(f"Model query failed: {str(e)}")

## Output Formatter

In [ ]:
def save_dataset(records, file_format: str, file_name: str):
    df = pd.DataFrame(records)
    print(df.shape)
    if file_format == ".csv":
        df.to_csv(file_name, index=False)
    elif file_format == ".json":
        df.to_json(file_name, orient="records", index=False)
    else:
        raise ValueError("Unsupported file format")

In [ ]:
def generate_dataset(
    model_name: str,
    business_case: str,
    num_records: int = 100,
    schema_text: str = None,
    file_format: str = '.csv',
    file_name: str = 'test_dataset.csv'
):
    """
    Generates a synthetic dataset using an LLM based on the given business case and optional schema.

    Returns:
        Tuple[str, pd.DataFrame | None]: A status message and a preview DataFrame (first 10 rows) if successful.
    """
    try:
        # Validate number of records
        if num_records <= 10:
            return "❌ Error: Number of records must be greater than 10.", None
        if num_records > 250:
            return "❌ Error: Number of records must be less than or equal to 250.", None

        # Validate file format
        if file_format not in FILE_FORMATS:
            return f"❌ Error: Invalid file format '{file_format}'. Supported formats: {FILE_FORMATS}", None

        # Ensure file name has correct extension
        if not file_name.endswith(file_format):
            file_name += file_format

        # Generate the prompt and query the model
        prompt = get_user_prompt(business_case, schema_text, num_records)
        print(prompt)
        print(model_name)
        records = query_llm(model_name, prompt)
        print(records)


        if not records:
            return "❌ Error: No valid records were generated by the model.", None

        # Save dataset
        save_dataset(records, file_format, file_name)

        # Prepare preview
        df = pd.DataFrame(records)
        preview = df.head(100)

        success_message = (
            f"✅ Generated {len(records)} records successfully!\n"
            f"📁 Saved to: {file_name}\n"
        )

        return success_message, preview

    except Exception as e:
        return f"❌ Error: {str(e)}", None

In [ ]:
with gr.Blocks(title="Synthetic Dataset Generator", theme=gr.themes.Monochrome()) as interface:
    tokenizer = None
    inputs = None
    hf_model = None
    outputs = None

    gr.Markdown("# Dataset Generator")
    gr.Markdown("Generate synthetic datasets using AI models")

    with gr.Row():
        with gr.Column(scale=2):
            schema_input = gr.Textbox(
                label="Schema",
                value=DEFAULT_SCHEMA_TEXT,
                lines=15,
                placeholder="Define your dataset schema here... Please follow this format: Name (TYPE) - Description, example: Example"
            )

            business_case_input = gr.Textbox(
                label="Business Case",
                value="I want to generate restaurant dataset",
                lines=1,
                placeholder="Enter business case description..."
            )

            with gr.Row():
                model_dropdown = gr.Dropdown(
                    label="Model",
                    choices=list(MODELS.keys()),
                    value=list(MODELS.keys())[0],
                    interactive=True
                )

                nr_records_input = gr.Number(
                    label="Number of records",
                    value=27,
                    minimum=11,
                    maximum=1000,
                    step=1
                )

            with gr.Row():
                filename_input = gr.Textbox(
                      label="Save as",
                      value="restaurant_dataset",
                      placeholder="Enter filename (extension will be added automatically)"
                  )

                file_format_dropdown = gr.Dropdown(
                    label="File format",
                    choices=FILE_FORMATS,
                    value=FILE_FORMATS[0],
                    interactive=True
                )

            generate_btn = gr.Button("🚀 Generate", variant="secondary", size="lg")

        with gr.Column(scale=1):
            gr.Markdown("""
            ### 📝 Dataset Generation Instructions

            1. **🗂 Schema** – Define your dataset structure
              *(default: restaurant schema provided)*
            2. **💡 Business Case** – Enter a prompt to guide the AI for generating data
            3. **🤖 Model** – Choose your AI model: GPT or Hugging Face
            4. **📊 Number of Records** – Specify entries to generate
              *(min: 11, max: 1000)*
            5. **📁 File Format** – Select output type: `.csv` or `.json`
            6. **💾 Save As** – Provide a filename *(extension auto-added)*
            7. **🚀 Generate** – Click **Generate** to create your dataset

            ### 🔧 Requirements

            Set API keys in Colab’s secret section:
              `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, `GOOGLE_API_KEY`, `HF_TOKEN`
            """)
            output_status = gr.Textbox(
                label="Status",
                lines=4,
                interactive=False
            )

            output_preview = gr.Dataframe(
                label="Preview (first 10 rows)",
                interactive=False,
                wrap=True
            )

    generate_btn.click(
        fn=generate_dataset,
        inputs=[
            model_dropdown,
            business_case_input,
            nr_records_input,
            schema_input,
            file_format_dropdown,
            filename_input
        ],
        outputs=[output_status, output_preview]
    )

interface.launch(debug=True)

del tokenizer, inputs, hf_model, outputs
gc.collect()
torch.cuda.empty_cache()

/tmp/ipykernel_543/2581831149.py:1: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Synthetic Dataset Generator", theme=gr.themes.Monochrome()) as interface:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://501d9ba959ac907091.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


The business case is: I want to generate restaurant dataset.
Generate 27 rows of data in CSV format.
Each line should be a CSV row with the following fields: 
1. Name (TEXT) - Name of the restaurant, example: Blue River Bistro
2. Address (TEXT) - Restaurant address, example: 742 Evergreen Terrace, Springfield, IL 62704
3. Type (TEXT) - Kitchen type, example: One of ["Thai","Mediterranean","Vegan","Steakhouse","Japanese"] or other potential types
4. Average Price (TEXT) - Average meal price, example: $45, or '--' if unknown
5. Year (INT) - Year of restaurant opening, example: 2015
6. Menu (Array) - List of meals, example: ["Grilled Salmon", "Caesar Salad", "Pad Thai", "Margherita Pizza", ...]

Phi 3 mini
nvidia/Mistral-NeMo-Minitron-8B-Instruct


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[]
